In [1]:
import pandas as pd
import requests
import os
import plotly.graph_objects as go
#nesse caso nao calcula reajuste
#from calculadora_inflacao import CalculadoraFatorInflacao

# APOGESP
## Cálculo de impacto orçamentário das propostas reajuste salarial para os APPGGS

Esse notebook realiza a estimativa de impacto orçamentário das propostas de reajuste salarial feitas pela APOGESP para a campanha salarial de 2026 na Prefeitura de São Paulo.

Nesse notebook, analisamos o impacto orçamentário da paridade com a proposta ora em estudo para os AMCI


Em ambos os casos, a estimativa segue a seguinte metodologia:

1. Identificação dos APPGGs em exercício (incluso cedidos) segundo os últimos dados disponíveis no portal de dados abertos (dezembro de 2025),
2. Atualização dos dados do item 1 considerando o acréscimo dos 30 APPGGs recém-nomeados no nível 1, criando dados sintéticos (novas linhas na tabela)
3. Cálculo do salário base atual de cada APPGG considerando a tabela atual e o nível em que ele se encontra,
4. Cálculo dos encargos e benefícios que tem como base o salário atual como contribuição previdenciária obrigatória, auxílio transporte, auxílio alimentação etc.
5. Cálculo do salário proposto considerando o nível em que o APPGG se encontra atualmente e a tabela atualizada proposta
6. Cálculo dos encargos citados no item 6,
7. Somatória dos salários e encargos para cada APPGG considerando a tabela atual e a tabela proposta,
8. Cálculo da diferença entre a situação atual e a situação proposta conforme item 7,
9. Somatória da diferença por nível da carreira APPGG (p. ex., para o nível 1 a proposta implica em X mil reais mensais para a Prefeitura para o nível 2, y mil reais etc.)
10. Simulação de progressão da carreira considerando 18 meses para a "subida" de nível e projeção do impacto orçamentário anual até o final da gestão (2028)

O diagrama na imagem abaixo representa as etapas da metodologia.

![Grafo da metodologia](mermaid_graph.svg)

## Dados servidores

Nessa seção pegamos os dados dos servidores ativos mais atualizados (julho de 2026) do portal de dados abertos. Em seguida, identificamos os APPGGs e adicionamos dados sintéticos referentes aos APPGGs recém-nomeados.

In [2]:
URL_SERVIDORES = 'https://dados.prefeitura.sp.gov.br/dataset/bf5df0f4-4fb0-4a5e-b013-07d098cc7b1c/resource/2e4c0ea6-3a9b-45c3-850e-0d7ff8a3ff2b/download/verificado_ativos_03-07-2026_jun-2026-1.csv'
file_dados = 'servidores_ativos_julho.csv'

In [3]:
def download_dados_servidores(fname:str, url:str=URL_SERVIDORES)->str:

    if not os.path.exists(fname):
        print('Downloading data')
        with requests.get(url) as r:
            r.raise_for_status()
            content = r.content
            with open(fname, 'wb') as f:
                f.write(content)
            
            assert os.path.exists(fname)
            return fname
    return fname

In [4]:
file_dados = download_dados_servidores(file_dados)

In [5]:
df = pd.read_csv(file_dados, encoding='latin1', sep=';')

In [6]:
df.head()

,REGISTRO,VINCULO,NOME,CARGO_BASICO,REF_CARGO_BAS,SEGMENTO,GRUPO,SUBGRUPO,ESCOL_CARGO_BASICO,CARGO_COMISSAO,...,SECRET_SUBPREF,SIGLA,SETOR,DISTRITO,SUBPREFEITURA,ORGAO_EXT,SEXO,ANO_NASCIMENTO,RACA_COR,PCD
0,1145541.0,16.0,CLEUZA BORGES PEREIRA SILVA,ASSESSOR IV,CDA-4,NaN,QC,CARGO EM COMISSAO,NAO SE APLICA,NaN,...,SECRETARIA MUNICIPAL DE GESTAO,SEGES,ASSESSORIA JURIDICA,SE,SE,NaN,FEMININO,1949.0,PARDA,NAO
1,1160206.0,5.0,ADILSON DA SILVA,COORDENADOR II,CDA-6,NaN,QC,CARGO EM COMISSAO,SUPERIOR COMPLETO,NaN,...,SUBPREFEITURA SANTO AMARO,SUB-SA,COORDENADORIA DE PLANEJAMENTO E DESENVOLVIMENT...,SANTO AMARO,SANTO AMARO,NaN,MASCULINO,1949.0,BRANCA,NAO
2,1160478.0,2.0,JULIO DE CARVALHO,FISCAL DE POSTURAS MUNICIPAIS NIVEL IV,QFPM15,NaN,QFPM,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SUBPREFEITURA SANTANA/TUCURUVI,SUB-ST,SUPERVISÃO TECNICA DE FISCALIZACAO,TUCURUVI,SANTANA/TUCURUVI,NaN,MASCULINO,1951.0,BRANCA,NAO
3,1161181.0,9.0,NEUSA PEDRAO NASSIR,ASSESSOR V,CDA-5,NaN,QC,CARGO EM COMISSAO,NAO SE APLICA,NaN,...,SECRETARIA DO GOVERNO MUNICIPAL,SGM,ASSESSORIA JURIDICA,SE,SE,NaN,FEMININO,1947.0,BRANCA,NAO
4,1168185.0,4.0,MARIA HELENA DI VERNIERI CUPPARI,ASSISTENTE TECNICO EDUCACIONAL,QPE17A,NaN,QPE L. 14660/07 RGPS,CARGO EM COMISSAO,LICENCIATURA PLENA COMPLETA,NaN,...,SECRETARIA MUNICIPAL DE EDUCACAO,SME,DIRETORIA REGIONAL DE EDUCACAO PIRITUBA/JARAGUA,LAPA,LAPA,NaN,FEMININO,1942.0,BRANCA,NAO


In [7]:
cargos = df['REF_CARGO_BAS'].unique()

In [8]:
for cargo in cargos:
    if "appgg" in str(cargo).lower():
        print(cargo)

APPGG3
APPGG1
APPGG2
APPGG5
APPGG6
APPGG4


In [9]:
#filtrando para apenas servidores efetivos (ou seja, que possuem cargo base)
df_efetivos = df[df['REF_CARGO_BAS'].notna()]

In [10]:
appggs = df_efetivos[df_efetivos['REF_CARGO_BAS'].str.contains('APPGG')].reset_index(drop=True)

In [11]:
appggs.head()

,REGISTRO,VINCULO,NOME,CARGO_BASICO,REF_CARGO_BAS,SEGMENTO,GRUPO,SUBGRUPO,ESCOL_CARGO_BASICO,CARGO_COMISSAO,...,SECRET_SUBPREF,SIGLA,SETOR,DISTRITO,SUBPREFEITURA,ORGAO_EXT,SEXO,ANO_NASCIMENTO,RACA_COR,PCD
0,6394604.0,3.0,CLAUDIO AGUIAR ALMEIDA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG3,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE CULTURA E ECONOMIA CRI...,SMC,SECRETARIA MUNICIPAL DE CULTURA E ECONOMIA CRI...,SE,SE,NaN,MASCULINO,1963.0,PRETA,NAO
1,7575491.0,2.0,MAURICIO DA SILVA CORREIA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG1,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE GESTAO,SEGES,SECRETARIA MUNICIPAL DE GESTAO,SE,SE,PREFEITURA DO MUNICIPIO DE ITAJAI,MASCULINO,1986.0,BRANCA,NAO
2,7718543.0,6.0,MARCIA MIYUKI ISHIKAWA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE HABITACAO,SEHAB,DEPARTAMENTO DE PLANEJAMENTO HABITACIONAL,SE,SE,NaN,FEMININO,1978.0,BRANCA,NAO
3,7794720.0,2.0,TIAGO ROSA MACHADO,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,ASSESSOR III,...,SECRETARIA MUNICIPAL DE ESPORTES E LAZER,SEME,SECRETARIA MUNICIPAL DE ESPORTES E LAZER,MOEMA,VILA MARIANA,NaN,MASCULINO,1983.0,BRANCA,NAO
4,7840501.0,2.0,THAIS ROBERTO DA SILVA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG5,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE DIREITOS HUMANOS E CID...,SMDHC,SECRETARIA MUNICIPAL DE DIREITOS HUMANOS E CID...,SE,SE,NaN,FEMININO,1977.0,PRETA,NAO


In [12]:
appggs = appggs[['REGISTRO', 'NOME', 'REF_CARGO_BAS', 'SIGLA', 'DATA_INICIO_EXERC']]

renomear_cols = {
    'REGISTRO' : 'rf',
    'NOME' : 'nome',
    'REF_CARGO_BAS' : 'cargo_base',
    'SIGLA' : 'secretaria_dez_2025',
    'DATA_INICIO_EXERC' : 'dt_inicio_exercicio'
}

appggs.rename(renomear_cols, axis=1, inplace=True)


In [13]:
appggs['nivel_carreira'] = appggs['cargo_base'].str.extract(r'(\d+)$').astype(int)

In [14]:
appggs.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira
0,6394604.0,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,01/10/2021,3
1,7575491.0,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,03/11/2021,1
2,7718543.0,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,08/12/2021,2
3,7794720.0,TIAGO ROSA MACHADO,APPGG2,SEME,05/01/2022,2
4,7840501.0,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,29/11/2017,5


In [15]:
appggs.shape

(191, 6)

In [16]:
#ESTOU CONSIDERANDO AS 43 NOMEAÇÕES APROVADAS

QTD_RECEM_NOMEADOS=43
recem_nomeados = [{
            'rf' : str(i+1).zfill(7),
            'nome' : f'recem_nomeado_{i+1}',
            'cargo_base' : 'APPGG1',
            'secretaria_dez_2025' : 'NA',
            'nivel_carreira' : 1,
            'dt_inicio_exercicio' : '01/03/2026'
            }
            for i in range(QTD_RECEM_NOMEADOS)
            ]

recem_nomeados = pd.DataFrame(recem_nomeados)

In [17]:
recem_nomeados.head()

,rf,nome,cargo_base,secretaria_dez_2025,nivel_carreira,dt_inicio_exercicio
0,0000001,recem_nomeado_1,APPGG1,NA,1,01/03/2026
1,0000002,recem_nomeado_2,APPGG1,NA,1,01/03/2026
2,0000003,recem_nomeado_3,APPGG1,NA,1,01/03/2026
3,0000004,recem_nomeado_4,APPGG1,NA,1,01/03/2026
4,0000005,recem_nomeado_5,APPGG1,NA,1,01/03/2026


In [18]:
recem_nomeados.shape

(43, 6)

In [19]:
appggs = pd.concat([appggs, recem_nomeados])

In [20]:
appggs.shape

(234, 6)

In [21]:
#vou colocar como datetime porque precisamos calcular com base na data
appggs['dt_inicio_exercicio'] = pd.to_datetime(appggs['dt_inicio_exercicio'], format ="%d/%m/%Y")


#inclusive ja vou definir se a pessoa contribui para o regime proprio (IPREM) ou nao
appggs['contribui_rpps'] = appggs['dt_inicio_exercicio'].dt.year<2018

In [22]:
appggs.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps
0,6394604.0,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False
1,7575491.0,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False
2,7718543.0,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False
3,7794720.0,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False
4,7840501.0,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True


# Situaçao atual

Nessa seção calculamos a situação atual de dispêndio com a carreira de APPGGs por parte da Prefeitura, considerando apenas os vencimentos e os encargos que estão relacionados aos vencimentos, como contribuição previdenciaria.

link para tabela usada: https://clic.prefeitura.sp.gov.br/storage/uploads/2026/06/10/Tabelas%20de%20Vencimentos_05_2026%20Clic%202026.pdf

In [23]:
tabela_atual = {
    1 : 13815.84,
    2 : 15197.43,
    3 : 15577.36,
    4 : 15966.79,
    5 : 16365.96,
    6 : 16775.11,
    7 : 18788.14,
    8 : 19257.84,
    9 : 19739.29,
    10 : 20232.76,
    11 : 20738.60,
    12 : 22875.96,
    13 : 23447.85,
    14 : 24034.05,
    15 : 24634.90,
}

In [24]:
appggs_atual = appggs.copy(deep=True)

In [25]:
appggs_atual['vencimento'] = appggs_atual['nivel_carreira'].map(tabela_atual)

In [26]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento
0,6394604.0,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False,15577.36
1,7575491.0,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13815.84
2,7718543.0,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,15197.43
3,7794720.0,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,15197.43
4,7840501.0,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,16365.96


### Terço adicional de férias e décimo terceiro

Nesse caso vamos dividir ambos por 12 são gastos anuais e nossa base é mensal.

In [27]:
def calcular_decimo_terceiro(row):

    return round(row['vencimento']/12, 2)

def calcular_terco_ferias(row):

    return round((row['vencimento']/3)/12, 2)

In [28]:
appggs_atual['decimo_terceiro'] = appggs_atual.apply(calcular_decimo_terceiro, axis=1)

appggs_atual['terco_ferias'] = appggs_atual.apply(calcular_terco_ferias, axis=1)

In [29]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias
0,6394604.0,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False,15577.36,1298.11,432.70
1,7575491.0,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13815.84,1151.32,383.77
2,7718543.0,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,15197.43,1266.45,422.15
3,7794720.0,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,15197.43,1266.45,422.15
4,7840501.0,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,16365.96,1363.83,454.61


### Vale alimentação

O vale alimentação só é devido para quem ganha menos de 10 salários minimos. Como com o reajuste essa proporcao pode diminuir, ele deve entrar na conta apesar de ser um valor fixo.

In [30]:
def calcular_va(row):
    SALARIO_MINIMO = 1621

    valores = {
        3 : 750.84,
        5 : 500.56,
        6 : 375.42,
        7 : 250.26,
        #tem que repetir no ultimo caso
        10 : 250.26
    }

    #acima de 10 sm nao recebe
    if row['vencimento'] >= (SALARIO_MINIMO*10):
        return 0
    
    for key, value in valores.items():
        if row['vencimento'] <= SALARIO_MINIMO*key:
            return value

    raise ValueError(f'Valor do vencimento {row["vencimento"]} nao se encaixa em nenhum caso')

In [31]:
appggs_atual['vale_alimentacao'] = appggs_atual.apply(calcular_va, axis=1)

In [32]:
def vale_refeicao(row):
    #apesar de ser uma constante decidi adicionar para os valores absolutos ficarem mais corretos
    DIAS_MES = 22
    VALOR_DIARIO  = 31.28

    return DIAS_MES*VALOR_DIARIO


In [33]:
appggs_atual['vale_refeicao'] = appggs_atual.apply(vale_refeicao, axis=1)

In [34]:
appggs_atual.sample(3)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao
10,0000011,recem_nomeado_11,APPGG1,NA,2026-03-01,1,False,13815.84,1151.32,383.77,250.26,688.16
131,8915504.0,GUSTAVO HENRIQUE MOREIRA ALVES,APPGG2,SME,2022-02-25,2,False,15197.43,1266.45,422.15,250.26,688.16
186,9537929.0,IGOR ROCHA DA SILVA,APPGG1,SMSUB,2026-02-06,1,False,13815.84,1151.32,383.77,250.26,688.16


### Contribuicao previdenciaria

Para calcular a contribuicao previdenciaria precisamos saber se a pessoa entrou antes de 2018 (ou seja, se é do IPREM) ou depois (se é do SAMPAPREV)

In [35]:
def valor_iprem(row):

    CONTRIBUICAO_IPREM = 0.28

    if not row['contribui_rpps']:
        return 0
    
    # o iprem nao considera o terço adicional de férias
    valor_base = row['vencimento'] + row['decimo_terceiro']
    
    return round(valor_base * CONTRIBUICAO_IPREM, 2)

In [36]:
appggs_atual['contribuicao_iprem'] = appggs_atual.apply(valor_iprem, axis=1)

In [37]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem
59,8359491.0,THIAGO FERREIRA DE SOUZA,APPGG6,SMSUB,2016-06-17,6,True,16775.11,1397.93,465.98,0.00,688.16,5088.45
2,0000003,recem_nomeado_3,APPGG1,NA,2026-03-01,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00
171,9445994.0,FELIPE LARA VOGEL,APPGG1,SMT,2024-10-21,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00
155,9411844.0,PAULO SEIKISHI HIGA,APPGG1,SEGES,2024-07-15,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00


In [38]:
def contribuicao_inss(row):

    ALIQUOTA_INSS = 0.21
    TETO_INSS = 8475.55

    if row['contribui_rpps']:
        return 0
    
    # o inss vai sobre todos os vencimentos, incluso terco adicional
    valor_base = row['vencimento'] + row['terco_ferias'] + row['decimo_terceiro']

    #só para validar o teto - no caso de APPGG da na mesma porque tá todo mundo acima, mas pra deixar a funcao correta
    if valor_base > TETO_INSS:
        valor_base = TETO_INSS
        
    return round(valor_base*ALIQUOTA_INSS, 2)
    


In [39]:
appggs_atual['contribuicao_inss'] = appggs_atual.apply(contribuicao_inss, axis=1)

In [40]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss
5,8038848.0,RODRIGO RAVACCI BRISOLA,APPGG1,SEME,2026-04-02,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87
185,9537911.0,LORENA STEPHANY LOPES MOREIRA DA CRUZ,APPGG1,SMADS,2026-02-06,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87
100,8896968.0,JOAO BONETT NETO,APPGG3,SMT,2021-10-14,3,False,15577.36,1298.11,432.70,250.26,688.16,0.00,1779.87
72,8414599.0,LOUISE RODRIGUES DE VASCONCELOS CORACINI,APPGG4,SEGES,2017-06-09,4,True,15966.79,1330.57,443.52,250.26,688.16,4843.26,0.00


In [41]:
def previdencia_complementar(row):

    TETO_INSS = 8475.55
    ALIQUOTA_COMPLEMENTAR = 0.075
    if row['contribui_rpps']:
        return 0
    
    valor_base = row['vencimento'] + row['decimo_terceiro']
    #só contribui o que é acima do teto
    valor_base = valor_base - TETO_INSS

    return round(valor_base * ALIQUOTA_COMPLEMENTAR, 2)



In [42]:
appggs_atual['previdencia_complementar'] = appggs_atual.apply(previdencia_complementar, axis=1)

In [43]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss,previdencia_complementar
79,8437904.0,ANDRE RICARDO MARTELINI,APPGG1,SMADS,2024-10-07,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87,486.87
166,9412018.0,DOUGLAS HENRIQUE DE SOUZA XAVIER,APPGG1,SEGES,2024-07-25,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87,486.87
98,8894329.0,BRUNO LUIS SIMOES GERALDO,APPGG3,SEGES,2021-09-22,3,False,15577.36,1298.11,432.70,250.26,688.16,0.00,1779.87,629.99
72,8414599.0,LOUISE RODRIGUES DE VASCONCELOS CORACINI,APPGG4,SEGES,2017-06-09,4,True,15966.79,1330.57,443.52,250.26,688.16,4843.26,0.00,0.00


In [44]:
def calculo_ir_simples(base_calculo:float)->float:
    '''Calcula o IR de forma simplificada considerando que todos os APPGGs ganham já acima da última faixa
    então não precisa aplicar redutor nem isenção etc.'''

    ultima_faixa = 7350
    if base_calculo <= ultima_faixa:
        raise ValueError('Base de cálculo deve ser maior que a última faixa para o cálculo simplificado.')

    aliquota = 0.275
    deducao = 908.73
    return round((base_calculo*aliquota)-deducao, 2)

def irpf_recolhido_na_fonte(row):

    #o IRPF é sobre vencimento + 3o adicional de ferias + 13o, ou seja, sobre tudo que é pago mensalmente

    base_calculo = row['vencimento'] + row['terco_ferias'] + row['decimo_terceiro']
    #considerando que todos os APPGGs ganham acima da última faixa, posso usar o cálculo simplificado
    return calculo_ir_simples(base_calculo)

In [45]:
appggs_atual['irpf_na_fonte'] = appggs_atual.apply(irpf_recolhido_na_fonte, axis=1)
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,irpf_na_fonte
144,9411704.0,LUCAS COTOSCK LARA,APPGG1,SEPLAN,2024-07-15,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87,486.87,3312.78
124,8915334.0,ANTONIO JOSE FARIA DA COSTA,APPGG2,SVMA,2022-01-18,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87,599.12,3734.93
36,8359008.0,MARCO AURELIO LESSA VILLELA,APPGG6,SEGES,2016-06-29,6,True,16775.11,1397.93,465.98,0.00,688.16,5088.45,0.00,0.00,4217.00
2,0000003,recem_nomeado_3,APPGG1,NA,2026-03-01,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87,486.87,3312.78


## Valor total

Agora calculamos o valor total e salvamos os dados

In [46]:
def valor_total_prefeitura(row):

    vencimentos = row['vencimento'] + row['decimo_terceiro'] + row['terco_ferias']
    auxilios = row['vale_alimentacao'] + row['vale_refeicao']
    previdencia = row['contribuicao_iprem'] + row['contribuicao_inss'] + row['previdencia_complementar']

    custos_totais = vencimentos + auxilios + previdencia
    #agora tem que tirar o irpf porque a prefeitura recolhe na fonte e já vai pro tesouro
    irpf = row['irpf_na_fonte']
    return round(custos_totais - irpf, 2)

In [47]:
appggs_atual['valor_total_prefeitura'] = appggs_atual.apply(valor_total_prefeitura, axis=1)

In [48]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,irpf_na_fonte,valor_total_prefeitura
0,6394604.0,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False,15577.36,1298.11,432.70,250.26,688.16,0.00,1779.87,629.99,3851.02,16805.43
1,7575491.0,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87,486.87,3312.78,15243.31
2,7718543.0,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87,599.12,3734.93,16468.51
3,7794720.0,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87,599.12,3734.93,16468.51
4,7840501.0,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,16365.96,1363.83,454.61,0.00,688.16,4964.34,0.00,0.00,4091.98,19744.92


In [49]:
appggs_atual.to_csv('situacao_atual.csv', sep=';')

# Paridade Proposta AMCI

In [50]:
tabela_atualizada = {
    1: 21527.50,
    2: 23680.25,
    3: 24745.86,
    4: 25859.43,
    5: 27023.10,
    6: 28239.14,
    7: 31063.05,
    8: 32150.26,
    9: 33275.52,
    10: 34440.16,
    11: 35645.57,
    12: 37427.85,
    13: 38363.54,
    14: 39130.81,
    15: 39987.00
}

In [51]:
for key, value in tabela_atualizada.items():
    assert value < 40000, "Valor acima do salario do Prefeito"
    assert value > tabela_atual[key], "Valor da tabela atualizada deve ser maior que o da tabela atual"
else:
    print("Dados da tabela consistentes. Nenhum valor acima do salário do Prefeito e todos os valores da tabela atualizada são maiores que os da tabela atual.")

Dados da tabela consistentes. Nenhum valor acima do salário do Prefeito e todos os valores da tabela atualizada são maiores que os da tabela atual.


In [52]:
proposta_amci = appggs.copy(deep=True)

Agora vamos calcular rapidamente o que fizemos na seção anterior, qual seja:
1. o novo vencimento,
2. o décimo terceiro,
3. o terço adicional de férias,
4. o auxilio alimentacao,
5. as contribuicoes previdenciarias
6. o irpf na fonte
7. o valor total

In [53]:
#vencimento
proposta_amci['vencimento'] = proposta_amci['nivel_carreira'].map(tabela_atualizada)
#ferias e decimo terceiro
proposta_amci['decimo_terceiro'] = proposta_amci.apply(calcular_decimo_terceiro, axis=1)
proposta_amci['terco_ferias'] = proposta_amci.apply(calcular_terco_ferias, axis=1)
#vale alimentacao
proposta_amci['vale_alimentacao'] = proposta_amci.apply(calcular_va, axis=1)
proposta_amci['vale_refeicao'] = proposta_amci.apply(vale_refeicao, axis=1)
#previdencia
proposta_amci['contribuicao_iprem'] = proposta_amci.apply(valor_iprem, axis=1)
proposta_amci['contribuicao_inss'] = proposta_amci.apply(contribuicao_inss, axis=1)
proposta_amci['previdencia_complementar'] = proposta_amci.apply(previdencia_complementar, axis=1)
#irpf
proposta_amci['irpf_na_fonte'] = proposta_amci.apply(irpf_recolhido_na_fonte, axis=1)
#VALOR TOTAL PROPOSTA
proposta_amci['valor_total_prefeitura'] = proposta_amci.apply(valor_total_prefeitura, axis=1)

In [54]:
proposta_amci.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,irpf_na_fonte,valor_total_prefeitura
0,6394604.0,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False,24745.86,2062.16,687.39,0,688.16,0.00,1779.87,1374.94,6652.51,24685.87
1,7575491.0,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,21527.50,1793.96,597.99,0,688.16,0.00,1779.87,1113.44,5669.12,21831.80
2,7718543.0,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,23680.25,1973.35,657.78,0,688.16,0.00,1779.87,1288.35,6326.90,23740.86
3,7794720.0,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,23680.25,1973.35,657.78,0,688.16,0.00,1779.87,1288.35,6326.90,23740.86
4,7840501.0,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,27023.10,2251.92,750.64,0,688.16,8197.01,0.00,0.00,7348.33,31562.50


In [55]:
proposta_amci.to_csv('proposta_amci_PL_agosto_2026.csv', sep=';')

In [56]:
tabela_atual

{1: 13815.84,
 2: 15197.43,
 3: 15577.36,
 4: 15966.79,
 5: 16365.96,
 6: 16775.11,
 7: 18788.14,
 8: 19257.84,
 9: 19739.29,
 10: 20232.76,
 11: 20738.6,
 12: 22875.96,
 13: 23447.85,
 14: 24034.05,
 15: 24634.9}

In [57]:
tabela_atualizada

{1: 21527.5,
 2: 23680.25,
 3: 24745.86,
 4: 25859.43,
 5: 27023.1,
 6: 28239.14,
 7: 31063.05,
 8: 32150.26,
 9: 33275.52,
 10: 34440.16,
 11: 35645.57,
 12: 37427.85,
 13: 38363.54,
 14: 39130.81,
 15: 39987.0}

In [58]:
def tabela_to_pandas(tabela:dict, tabela_name:str):

    dados = []
    for nivel, val in tabela.items():
        dados.append([tabela_name, nivel, val])

    return pd.DataFrame(dados, columns = ['nome_tabela', 'nivel', 'vencimento'])

In [59]:
tabelas = {
    'atual' : tabela_atual,
    'original_atualizada_amci' : tabela_atualizada,
}

In [60]:
dfs_tabelas = [tabela_to_pandas(tabela, tabela_name) for tabela_name, tabela in tabelas.items()]
df_tabelas = pd.concat(dfs_tabelas)

df_tabelas

,nome_tabela,nivel,vencimento
0,atual,1,13815.84
1,atual,2,15197.43
2,atual,3,15577.36
3,atual,4,15966.79
4,atual,5,16365.96
5,atual,6,16775.11
6,atual,7,18788.14
7,atual,8,19257.84
8,atual,9,19739.29
9,atual,10,20232.76


# Agregações

Agora fazemos as agregações para analisar o impacto orçamentário

In [61]:
def total_mensal_por_nivel(servidores, nome_proposta:str):

    df = servidores.groupby('nivel_carreira')['valor_total_prefeitura'].sum().reset_index()
    df['id_proposta'] = nome_proposta
    return df

In [62]:
total_atual = total_mensal_por_nivel(appggs_atual, 'situacao_atual')
total_atual

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,1831121.26,situacao_atual
1,2,576397.85,situacao_atual
2,3,456061.88,situacao_atual
3,4,19552.55,situacao_atual
4,5,256683.96,situacao_atual
5,6,767547.94,situacao_atual


In [63]:
mensal_atual = total_atual['valor_total_prefeitura'].sum()
mensal_atual

np.float64(3907365.4399999995)

In [64]:
anual_atual = mensal_atual*12
anual_atual

np.float64(46888385.279999994)

In [65]:
total_amci = total_mensal_por_nivel(proposta_amci, 'proposta_amci')
total_amci

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,2623452.70,proposta_amci
1,2,830930.10,proposta_amci
2,3,670869.93,proposta_amci
3,4,30272.13,proposta_amci
4,5,410312.50,proposta_amci
5,6,1250616.10,proposta_amci


In [66]:
total_amci_mensal = total_amci['valor_total_prefeitura'].sum()
total_amci_mensal

np.float64(5816453.459999999)

In [67]:
total_amci_anual = total_amci_mensal*12
total_amci_anual

np.float64(69797441.51999998)

In [68]:
total_amci_anual-anual_atual

np.float64(22909056.239999987)

In [69]:
def impacto_mensal(situacao_atual, proposta):

    join = pd.merge(situacao_atual, proposta, on='nivel_carreira', suffixes=('_atual', '_proposta'))
    join['impacto_mensal'] = join['valor_total_prefeitura_proposta'] - join['valor_total_prefeitura_atual']

    return join



In [70]:
impacto_mensal_amci = impacto_mensal(total_atual, total_amci)


In [71]:
impacto_mensal_amci

,nivel_carreira,valor_total_prefeitura_atual,id_proposta_atual,valor_total_prefeitura_proposta,id_proposta_proposta,impacto_mensal
0,1,1831121.26,situacao_atual,2623452.70,proposta_amci,792331.44
1,2,576397.85,situacao_atual,830930.10,proposta_amci,254532.25
2,3,456061.88,situacao_atual,670869.93,proposta_amci,214808.05
3,4,19552.55,situacao_atual,30272.13,proposta_amci,10719.58
4,5,256683.96,situacao_atual,410312.50,proposta_amci,153628.54
5,6,767547.94,situacao_atual,1250616.10,proposta_amci,483068.16


In [72]:
impacto_anual = round(impacto_mensal_amci['impacto_mensal'].sum()*12, 2)

In [73]:
impacto_anual

np.float64(22909056.24)

In [74]:
custo_anual_ipc = round(total_amci['valor_total_prefeitura'].sum()*12, 2)

In [75]:
custo_anual_ipc

np.float64(69797441.52)

In [76]:
impactos_mensais = {'paridade_amci' : impacto_mensal_amci}

In [77]:

def gerar_grafico_niveis_df(df, col_valor:str, titulo:str, nome_arquivo="grafico_niveis.png"):
    
    col_nivel = 'nivel_carreira'
    
    cores = ["steelblue"] * len(df)
    indice_maximo = df[col_valor].idxmax()
    cores[indice_maximo] = "crimson"

    # Formatação para o texto sobre as barras (R$ 1.234,56)
    texto_formatado = df[col_valor].apply(
        lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    )

    fig = go.Figure(data=[
        go.Bar(
            x=df[col_nivel],
            y=df[col_valor],
            text=texto_formatado,
            textposition="outside",
            marker_color=cores
        )
    ])

    fig.update_layout(
        title=dict(
            text=titulo,
            x=0.5,
            xanchor="center",
            font=dict(size=22)
        ),
        # Define os separadores globalmente: o primeiro é o decimal, o segundo é o de milhar
        separators=",.",
        plot_bgcolor="white",
        width=1200,
        height=600,
        xaxis=dict(
            title="Níveis de Carreira",
            tickangle=-45,
            categoryorder="array",
            categoryarray=df[col_nivel].tolist()
        ),
        yaxis=dict(
            title="Valores em R$",
            showgrid=True,
            gridcolor="lightgrey",
            # Formata os números do eixo Y com separador de milhar e 2 casas decimais
            tickformat=",2f"
        ),
        margin=dict(l=50, r=50, t=100, b=120)
    )

    fig.write_image(nome_arquivo)

    return fig

In [78]:
import kaleido

# Baixa ou localiza o Chrome automaticamente
kaleido.get_chrome()

<coroutine object get_chrome at 0x7c58f862f5b0>

In [79]:

for nome, df in impactos_mensais.items():
    titulo = f"Custo total por nível: {nome}"
    nome_arquivo = f"grafico_niveis_{nome.replace(' ', '_').lower()}.png"
    gerar_grafico_niveis_df(df, "valor_total_prefeitura_proposta", titulo, nome_arquivo)

In [80]:
for proposta, df in impactos_mensais.items():
    valor_total = df['valor_total_prefeitura_proposta'].sum()*12
    valor_formatado = f"R$ {valor_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    print(f"Valor total anual: {valor_formatado} - {proposta}")

Valor total anual: R$ 69.797.441,52 - paridade_amci


In [81]:
for proposta, df in impactos_mensais.items():
    try:
        valor_total = df['impacto_mensal'].sum()*12
        valor_formatado = f"R$ {valor_total:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        print(f"Impacot total anual: {valor_formatado} - {proposta}")
    except KeyError:
        print(proposta, "não possui impacto")

Impacot total anual: R$ 22.909.056,24 - paridade_amci


In [82]:
niveis: DataFrame = df_tabelas

In [83]:
niveis

,nome_tabela,nivel,vencimento
0,atual,1,13815.84
1,atual,2,15197.43
2,atual,3,15577.36
3,atual,4,15966.79
4,atual,5,16365.96
5,atual,6,16775.11
6,atual,7,18788.14
7,atual,8,19257.84
8,atual,9,19739.29
9,atual,10,20232.76


In [84]:
import plotly.graph_objects as go
import pandas as pd

def exportar_graficos_comparativos(df, col_valor="vencimento"):
    col_nivel = "nivel"
    col_tabela = "nome_tabela"
    
    # Identifica as tabelas que serão comparadas com a 'atual'
    tabelas_extras = [t for t in df[col_tabela].unique() if t != "atual"]
    
    for tabela in tabelas_extras:
        # Filtra apenas o par necessário
        df_par = df[df[col_tabela].isin(["atual", tabela])]
        
        fig = go.Figure()

        # Adiciona as barras para 'atual' e para a 'tabela' da vez
        for nome in ["atual", tabela]:
            df_sub = df_par[df_par[col_tabela] == nome]
            
            # Formatação Real (sem centavos para evitar sobreposição de texto)
            texto = df_sub[col_valor].apply(
                lambda x: f"R$ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")
            )

            fig.add_trace(go.Bar(
                x=df_sub[col_nivel],
                y=df_sub[col_valor],
                name=nome.replace("_", " ").title(),
                text=texto,
                textposition="outside",
                marker_color="steelblue" if nome == "atual" else "crimson"
            ))

        fig.update_layout(
            title=dict(
                text=f"Comparativo: Atual vs {tabela.replace('_', ' ').title()}",
                x=0.5,
                font=dict(size=22)
            ),
            barmode="group",
            separators=",.",
            plot_bgcolor="white",
            width=1200,
            height=600,
            xaxis=dict(title="Nível de Carreira", type="category"),
            yaxis=dict(
                title="Vencimento (R$)",
                showgrid=True,
                gridcolor="lightgrey",
                tickformat=",0f"
            ),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
            margin=dict(l=50, r=50, t=100, b=50)
        )

        # Salva cada gráfico com o nome da tabela correspondente
        nome_arquivo = f"comparativo_atual_vs_{tabela}.png"
        fig.write_image(nome_arquivo, engine="kaleido")
        print(f"Arquivo salvo: {nome_arquivo}")

# Exemplo de uso:
# exportar_graficos_comparativos(df)

In [85]:
exportar_graficos_comparativos(niveis)

/tmp/ipykernel_24171/3595305641.py:59: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image(nome_arquivo, engine="kaleido")


Arquivo salvo: comparativo_atual_vs_original_atualizada_amci.png


In [86]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,vale_refeicao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,irpf_na_fonte,valor_total_prefeitura
0,6394604.0,CLAUDIO AGUIAR ALMEIDA,APPGG3,SMC,2021-10-01,3,False,15577.36,1298.11,432.70,250.26,688.16,0.00,1779.87,629.99,3851.02,16805.43
1,7575491.0,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13815.84,1151.32,383.77,250.26,688.16,0.00,1779.87,486.87,3312.78,15243.31
2,7718543.0,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87,599.12,3734.93,16468.51
3,7794720.0,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,15197.43,1266.45,422.15,250.26,688.16,0.00,1779.87,599.12,3734.93,16468.51
4,7840501.0,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,16365.96,1363.83,454.61,0.00,688.16,4964.34,0.00,0.00,4091.98,19744.92


In [87]:
appggs_atual['nome'].str.startswith('recem_nomeado').sum()

np.int64(43)

In [88]:
qtd_por_nivel: DataFrame = appggs_atual.groupby('nivel_carreira').count()[['rf']].rename({'rf' : 'Quantidade'}, axis=1)

In [89]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(
        x=qtd_por_nivel.index.astype(str),  # Convertendo o índice para string para melhor exibição
        y=qtd_por_nivel['Quantidade'],
        text=qtd_por_nivel['Quantidade'],
        textposition='outside',
        marker_color='steelblue'
    )
])

fig.update_layout(
    title=dict(
        text="Quantidade por Nível na Carreira",
        x=0.5,
        font=dict(size=22)
    ),
    xaxis=dict(
        title="Nível na Carreira",
        tickangle=0
    ),
    yaxis=dict(
        title="Quantidade",
        showgrid=True,
        gridcolor="lightgrey"
    ),
    plot_bgcolor="white",
    width=800,
    height=500,
    margin=dict(l=50, r=50, t=100, b=50)
)
fig.write_image('quantidade_pessoas_por_nivel.png', engine="kaleido")

fig.show()

/tmp/ipykernel_24171/2642592332.py:33: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig.write_image('quantidade_pessoas_por_nivel.png', engine="kaleido")
